# cfbmodel on Colab

**There is no import order to worry about** — Python resolves imports itself. Two things
actually matter on Colab, and both are handled below:

1. **Folder structure must survive.** `cfbmodel/` has to stay a *directory*. If you upload
   the `.py` files individually into `/content`, `from cfbmodel import ...` fails, because
   there is no package for Python to find. Upload the zip, don't upload loose files.
2. **Colab wipes local disk on disconnect.** Re-pulling a season costs ~35 API calls against
   a 1,000/month free budget. Cell 2 points the cache at Drive so you pay once, ever.

Run the cells top to bottom the first time. After that only cells 1-3 and whatever you're
working on.


---
## 1 · Upload and unpack

Run this, then pick `cfbmodel.zip` in the file chooser.


In [ ]:
from google.colab import files
import zipfile, os, shutil

up = files.upload()                       # choose cfbmodel.zip
name = next(iter(up))

if os.path.exists('/content/cfbmodel'):
    shutil.rmtree('/content/cfbmodel')
with zipfile.ZipFile(name) as z:
    z.extractall('/content')

os.chdir('/content/cfbmodel')             # cwd matters — run_slate.py expects it
print(os.getcwd())
print(sorted(os.listdir('cfbmodel')))


### Alternative: keep the whole project on Drive

Better if you'll work on this across many sessions — edits survive, not just the cache.
Unzip once into Drive, then every future session is just the two lines in the second block.


In [ ]:
# --- one time only ---
# from google.colab import drive; drive.mount('/content/drive')
# import zipfile
# with zipfile.ZipFile('cfbmodel.zip') as z:
#     z.extractall('/content/drive/MyDrive/')

# --- every session after that ---
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.chdir('/content/drive/MyDrive/cfbmodel')


---
## 2 · Persist the cache to Drive

**Don't skip this.** The CFBD free tier is 1,000 calls/month. A full pull of five seasons
is roughly 175 calls. Without this cell you spend that again every time the runtime
disconnects, and you'll be rate-limited by October.

`CFBMODEL_CACHE` must be set *before* importing cfbmodel — the paths are resolved at
import time.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.environ['CFBMODEL_CACHE']  = '/content/drive/MyDrive/cfbmodel_data/cache'
os.environ['CFBMODEL_OUTPUT'] = '/content/drive/MyDrive/cfbmodel_data/output'

from cfbmodel.config import CACHE, OUTPUT
print('cache  ->', CACHE)
print('output ->', OUTPUT)
print('cached files already on Drive:', len(list(CACHE.glob('*.json'))))


---
## 3 · API key

Use Colab's secrets manager (🔑 in the left sidebar), not a hardcoded string — notebooks
get shared and re-shared. Add a secret named `CFBD_API_KEY`, toggle notebook access on.

Free key: https://collegefootballdata.com/key


In [ ]:
from google.colab import userdata
import os

os.environ['CFBD_API_KEY'] = userdata.get('CFBD_API_KEY')
print('key loaded:', bool(os.environ['CFBD_API_KEY']))


---
## 4 · Verify before you spend any API calls

Colab already has numpy, pandas, scipy, sklearn and requests, so `requirements.txt` is
effectively a no-op — but run the tests anyway. They use synthetic data and need no key,
so a failure here is a broken upload, not a broken model.


In [ ]:
!python tests/smoke_test.py
!python tests/test_upgrades.py


---
## 5 · Pull data  *(~175 API calls, once)*

Everything lands in the Drive cache. Re-runs cost zero.

Takes 10-20 minutes. Colab may idle-disconnect during it — that's survivable now,
since completed pulls are already on Drive. Just re-run; it skips what it has.


In [ ]:
!python run_slate.py pull --seasons 2021 2022 2023 2024 2025


---
## 6 · Validate — read this before betting anything

This is the only cell whose output should change your behaviour. It asks whether your
ratings explain any margin variance the closing line hasn't already priced.

- `model_margin` p > 0.10 → no signal. Don't bet sides or totals. The props path still stands.
- p < 0.05 → the coefficient *is* your `--w-model`. A coef of 0.15 means 15% model, 85% market.
- coef > 0.5 → almost certainly a lookahead leak. Find it before you trust anything.

Then read the **segmented** tables underneath. Aggregate results hide the whole story —
a model can be worthless against the Saturday P4 closer and live against a Tuesday MAC opener.


In [ ]:
!python run_slate.py validate --seasons 2022 2023 2024 2025


---
## 6b · Week 0, one command

Does the whole chain — pull, preseason ratings, project, compare to market, bet card.
~40 API calls the first time, zero on re-runs.

Week 0 is the only week where neither you nor the market has current-season data. That
symmetry is what makes weeks 0-3 the softest stretch of the calendar. It's also why every
number this prints is a prior with a wide distribution around it — which is how the pmf
prices it, as long as you don't read the point estimate as a forecast.


In [ ]:
!python run_week0.py --season 2026 --week 0


Sanity-check the top-15 preseason list against your own eyes before anything else.
If a team you know is bad shows up there, the input is wrong — not the sport.

A mean spread disagreement above ~6 points means the ratings are broken, not that you
found eight edges. The script says so itself if it happens.


---
## 7 · Weekly workflow

Set `W_MODEL` from cell 6's coefficient, not from how the numbers feel.


In [ ]:
SEASON, WEEK = 2026, 0
W_MODEL = 0.0    # <-- from cell 6. Leave at 0 until validate justifies otherwise.

!python run_slate.py preseason --season {SEASON}


In [ ]:
# Friday: totals-relevant forecasts. Confidence under ~0.7 means the
# adjustment is mostly noise and the market hasn't priced it either.
!python run_slate.py weather --season {SEASON} --week {WEEK}


In [ ]:
!python run_slate.py slate --season {SEASON} --week {WEEK} --w-model {W_MODEL}


In [ ]:
HOME, AWAY = 'Auburn', 'Georgia'

# --p-play 0.5 for a genuine game-time decision. CFB has no injury report;
# betting a questionable starter at face value is the most common way to lose here.
!python run_slate.py props --season {SEASON} --week {WEEK} --home '{HOME}' --away '{AWAY}'


---
## 8 · Working interactively instead of via CLI

The CLI is convenient but opaque. For actual research, import the modules and inspect
frames directly — this is where a notebook earns its keep over a terminal.


In [ ]:
import pandas as pd, numpy as np
from cfbmodel import ingest, ratings, game_model, script, qb, edge, backtest

SEASON, WEEK = 2026, 5

lines = pd.concat([ingest.lines(y) for y in (2024, 2025)], ignore_index=True)
mkt   = ratings.fit_market_ratings(lines, SEASON, WEEK)
mkt.sort_values('market_rating', ascending=False).head(20)


In [ ]:
# What a prop looks like under two different game scripts — the thing a
# static model cannot show you.
sc = script.simulate_game_script(exp_margin_for_team=24.0, exp_total=55.0,
                                 base_pace=68.0, n_sims=40_000)
rb = script.simulate_rush_yards_joint(carry_share=0.55, script=sc,
                                      ypc=5.1, ypc_sd=6.4, shock_sigma=0.18)

close = np.abs(sc['margin']) < 14
blow  = np.abs(sc['margin']) >= 28
print(f'overall      {rb.mean():6.1f}')
print(f'stays close  {rb[close].mean():6.1f}')
print(f'blowout      {rb[blow].mean():6.1f}')
print(f'benched in   {sc["p_pulled"]*100:.0f}% of sims')


In [ ]:
# Bet card for one prop. Props run 6-9% hold — a projection that beats the
# line by 3 yards is usually still negative EV once the vig is stripped.
from cfbmodel.props import price_prop

probs = price_prop(rb, line=84.5)
edge.evaluate_prop(probs, line=84.5, over_odds=-115, under_odds=-115,
                   player='RB1', market='rush_yds')


---
## Colab gotchas worth knowing

| symptom | cause | fix |
|---|---|---|
| `ModuleNotFoundError: cfbmodel` | files uploaded loose, or wrong cwd | re-run cell 1; the package must be a *directory* |
| cache empty after reconnect | `CFBMODEL_CACHE` set after import | restart runtime, run cell 2 *before* any cfbmodel import |
| `RuntimeError: CFBD_API_KEY not set` | secret not enabled for this notebook | 🔑 sidebar → toggle notebook access |
| pull dies partway | idle disconnect | just re-run — finished pulls are cached |
| edits to `.py` files don't take effect | Python caches imported modules | `%load_ext autoreload` / `%autoreload 2`, or Runtime → Restart |

Free-tier Colab disconnects after ~90 min idle and caps sessions around 12 hours. Nothing
here needs a GPU — the ridge fits run in seconds on CPU, so don't waste a GPU runtime on it.
